# Agent: Control (Phase 6)

This notebook consumes camera decisions and gate events, then publishes control commands and occupancy updates.

Phase 6 scope:
- Keep command logic in a dedicated control agent notebook
- Publish latest-command-wins commands with TTL metadata
- Retry command publishes under QoS 0 on a fixed interval
- Publish occupancy state periodically

## Run checklist
- Run Cells 2-4 to initialize settings and connect MQTT.
- Confirm Cell 4 prints a unique client suffix.
- Run Cell 5 to activate decision/event subscriptions.
- Keep Cell 6 running to publish commands and occupancy updates.

In [1]:
# Cell purpose: import modules and load shared config from project root
from __future__ import annotations

from datetime import datetime, timezone
import json
import os
from pathlib import Path
import sys
import time

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / 'src'
    if (src_dir / 'simulated_city').exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

os.environ['SIMCITY_MQTT_PROFILES'] = 'local'

from simulated_city.config import load_config
import simulated_city.mqtt as mqtt
from simulated_city.agent_rules import CommandEnvelope, LatestCommandStore, should_retry

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

print(f'Loaded config. MQTT base topic: {mqtt_cfg.base_topic}')

Loaded config. MQTT base topic: simulated-city/stadium


In [2]:
# Cell purpose: define phase-6 topic names, reliability settings, and local state stores
camera_decision_topic = f'{mqtt_cfg.base_topic}/camera/decision'
entry_event_topic = f'{mqtt_cfg.base_topic}/entry/event'
control_command_topic = f'{mqtt_cfg.base_topic}/control/command'
occupancy_topic = f'{mqtt_cfg.base_topic}/occupancy/state'

COMMAND_TTL_S = sim_cfg.command_timeout_s if sim_cfg else 5.0
COMMAND_RETRY_INTERVAL_S = sim_cfg.command_retry_interval_s if sim_cfg else 2.0
OCCUPANCY_PUBLISH_INTERVAL_S = sim_cfg.occupancy_publish_interval_s if sim_cfg else 2.0
HEARTBEAT_INTERVAL_S = sim_cfg.heartbeat_interval_s if sim_cfg else 10.0

command_store = LatestCommandStore()
command_sequence_by_person: dict[str, int] = {}
pending_commands: dict[str, dict] = {}
last_retry_ts_by_person: dict[str, float | None] = {}
inside_count = 0

received_decisions = 0
accepted_commands = 0
published_commands_ok = 0
published_commands_fail = 0

print(f'Subscribe: {camera_decision_topic}')
print(f'Subscribe: {entry_event_topic}')
print(f'Publish command: {control_command_topic}')
print(f'Publish occupancy: {occupancy_topic}')

Subscribe: simulated-city/stadium/camera/decision
Subscribe: simulated-city/stadium/entry/event
Publish command: simulated-city/stadium/control/command
Publish occupancy: simulated-city/stadium/occupancy/state


In [3]:
# Cell purpose: connect MQTT client for control-agent subscriptions and publishing
control_client_suffix = f"agent-control-phase6-{int(time.time())}"
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix=control_client_suffix)

print(f'Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}')
print(f'Control client suffix: {control_client_suffix}')

Connected to MQTT broker at 127.0.0.1:1883
Control client suffix: agent-control-phase6-1772629297


In [4]:
# Cell purpose: register callbacks that enforce latest-command-wins with TTL envelopes
def _to_command_payload(person_id: str, allowed: bool) -> dict:
    now_ts = time.time()
    sequence = command_sequence_by_person.get(person_id, 0) + 1
    command_sequence_by_person[person_id] = sequence
    command_name = 'enter' if allowed else 'leave'

    envelope = CommandEnvelope(
        person_id=person_id,
        command=command_name,
        sequence=sequence,
        issued_ts=now_ts,
        expires_ts=now_ts + COMMAND_TTL_S,
    )

    applied = command_store.upsert(envelope)
    if not applied:
        return {}

    active = command_store.get_active(person_id, now_ts)
    if active is None:
        return {}

    return {
        'person_id': active.person_id,
        'command': active.command,
        'sequence': active.sequence,
        'issued_ts': active.issued_ts,
        'expires_ts': active.expires_ts,
        'ttl_s': COMMAND_TTL_S,
        'source': 'control-agent',
    }

def on_message(client, userdata, msg):
    global inside_count, received_decisions, accepted_commands

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        return

    if msg.topic == camera_decision_topic:
        person_id = str(payload.get('person_id', '')).strip()
        if not person_id:
            return
        received_decisions += 1

        command_payload = _to_command_payload(person_id, bool(payload.get('allowed', False)))
        if command_payload:
            pending_commands[person_id] = command_payload
            last_retry_ts_by_person[person_id] = None
            accepted_commands += 1

    elif msg.topic == entry_event_topic:
        event_type = str(payload.get('event_type', '')).lower()
        if event_type == 'entered':
            inside_count += 1
        elif event_type == 'exited':
            inside_count = max(0, inside_count - 1)

client.on_message = on_message
client.subscribe(camera_decision_topic, qos=0)
client.subscribe(entry_event_topic, qos=0)

print('Control agent subscriptions active.')

Control agent subscriptions active.


In [ ]:
# Cell purpose: run publish loop with retry scheduling and periodic occupancy state publish
last_occupancy_publish_ts = 0.0
last_heartbeat_ts = 0.0

print('Control agent running. Press Interrupt to stop.')
while True:
    now_ts = time.time()

    for person_id, command_payload in list(pending_commands.items()):
        expires_ts = float(command_payload.get('expires_ts', 0.0))
        if now_ts > expires_ts:
            pending_commands.pop(person_id, None)
            last_retry_ts_by_person.pop(person_id, None)
            continue

        last_attempt = last_retry_ts_by_person.get(person_id)
        if not should_retry(last_attempt, now_ts, COMMAND_RETRY_INTERVAL_S):
            continue

        ok = mqtt.publish_json_checked(client, control_command_topic, command_payload, qos=0)
        last_retry_ts_by_person[person_id] = now_ts
        if ok:
            published_commands_ok += 1
            pending_commands.pop(person_id, None)
            last_retry_ts_by_person.pop(person_id, None)
        else:
            published_commands_fail += 1

    if now_ts - last_occupancy_publish_ts >= OCCUPANCY_PUBLISH_INTERVAL_S:
        occupancy_payload = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'inside_count': inside_count,
            'pending_commands': len(pending_commands),
        }
        mqtt.publish_json_checked(client, occupancy_topic, occupancy_payload, qos=0)
        last_occupancy_publish_ts = now_ts

    if now_ts - last_heartbeat_ts >= HEARTBEAT_INTERVAL_S:
        print(
            f'control heartbeat -> decisions={received_decisions} accepted={accepted_commands} '
            f'pending={len(pending_commands)} publish_ok={published_commands_ok} publish_fail={published_commands_fail} '
            f'inside_count={inside_count}'
        )
        last_heartbeat_ts = now_ts

    time.sleep(0.2)

In [ ]:
print({"received_decisions": received_decisions, "accepted_commands": accepted_commands, "pending_commands": len(pending_commands), "published_commands_ok": published_commands_ok, "published_commands_fail": published_commands_fail, "inside_count": inside_count})